# Omni-Embed-Audio (OEA) — Encoding Demo

This notebook loads an OEA checkpoint from the Hugging Face Hub, encodes the five bundled Clotho samples under `examples/data/clotho_samples/`, and computes cosine similarities against five query formulations (one per UIQ type: question, imperative, paraphrase, tagging, negative).

**Available checkpoints** (https://huggingface.co/JudeJiwoo):
`OEA-Qwen3B-Cl`, `OEA-Qwen3B-AC`, `OEA-Qwen7B-Cl`, `OEA-Qwen7B-AC`, `OEA-Nemo3B-Cl`, `OEA-Nemo3B-AC` (Cl=Clotho-trained LoRA+heads, AC=AudioCaps-trained).

**Audio sources:** the five WAVs are from Clotho v2 (Drossos et al., 2020), originally on Freesound. See `examples/data/clotho_samples/captions.jsonl` for the original Freesound filenames, IDs, uploaders, and per-clip licenses (CC0 / CC-BY 3.0).

## 1. Setup

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import json, sys
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import torch

REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

MODEL_NAME = 'OEA-Qwen3B-Cl'   # one of: OEA-{Qwen3B,Qwen7B,Nemo3B}-{Cl,AC}
CHECKPOINT_FILE = 'step_40.pt'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

BASE_MODEL = {
    'OEA-Nemo3B-Cl': 'nvidia/omni-embed-nemotron-3b',
    'OEA-Nemo3B-AC': 'nvidia/omni-embed-nemotron-3b',
    'OEA-Qwen3B-Cl': 'Qwen/Qwen2.5-Omni-3B',
    'OEA-Qwen3B-AC': 'Qwen/Qwen2.5-Omni-3B',
    'OEA-Qwen7B-Cl': 'Qwen/Qwen2.5-Omni-7B',
    'OEA-Qwen7B-AC': 'Qwen/Qwen2.5-Omni-7B',
}
print('base model :', BASE_MODEL[MODEL_NAME])
print('checkpoint :', f'JudeJiwoo/{MODEL_NAME}/{CHECKPOINT_FILE}')
print('device     :', DEVICE)

## 2. Bundled Clotho samples

All five clips are 30 s (the Clotho cap) and were chosen for diversity in sound type and a permissive license (CC0 / CC-BY 3.0). The first caption shown below is `caption_1` from the Clotho v2 evaluation set.

In [ ]:
audio_dir = REPO_ROOT / 'examples' / 'data' / 'clotho_samples'
manifest = audio_dir / 'captions.jsonl'

samples = [json.loads(line) for line in manifest.open(encoding='utf-8')]
audio_paths = [audio_dir / s['file'] for s in samples]
captions    = [s['captions'][0] for s in samples]

for s in samples:
    print(f"{s['file']:<26s} | {s['captions'][0]}")
    print(f"  Freesound #{s['freesound_sound_id']} · {s['freesound_uploader']} · {s['license']}")

## 3. Load OEA model (base + LoRA + projection heads)

OEA shares a single multimodal-LLM backbone for both modalities. The released checkpoint contains:
- `lora_state_dict` — LoRA adapters attached to attention layers,
- `audio_head`, `text_head` — 512-d projection heads with L2-normalized output.

In [ ]:
from huggingface_hub import hf_hub_download
from AudioRetrieval.models.omni_embed_adapter import OmniEmbedAdapter
from AudioRetrieval.training.oea.train_omniembed_lora import ProjectionHead, attach_lora

ckpt_path = hf_hub_download(repo_id=f'JudeJiwoo/{MODEL_NAME}', filename=CHECKPOINT_FILE)

adapter = OmniEmbedAdapter(
    repo_id=BASE_MODEL[MODEL_NAME],
    device=DEVICE,
    passage_prefix='passage:',
    query_prefix='query:',
)

lora_cfg = SimpleNamespace(
    lora_rank=16, lora_alpha=32, lora_dropout=0.05,
    lora_targets=['q_proj','k_proj','v_proj','o_proj','qkv','out_proj'],
)
peft_model = attach_lora(adapter.get_underlying_model(), lora_cfg)
adapter.set_underlying_model(peft_model)

device = torch.device(DEVICE)
ckpt = torch.load(ckpt_path, map_location=device)
peft_model.load_state_dict(ckpt['lora_state_dict'], strict=False)

hidden = peft_model.config.text_config.hidden_size
audio_head = ProjectionHead(hidden, 512, 0.1).to(device).eval()
text_head  = ProjectionHead(hidden, 512, 0.1).to(device).eval()
audio_head.load_state_dict(ckpt['audio_head'])
text_head.load_state_dict(ckpt['text_head'])
print('model loaded')

## 4. Encode the five samples and five UIQ-style queries

In [ ]:
def encode_audio(paths):
    raw = adapter.encode_audio([str(p) for p in paths], batch_size=4)
    with torch.inference_mode():
        return audio_head(torch.from_numpy(raw).to(device)).cpu().float().numpy()

def encode_text(texts):
    raw = adapter.encode_text(texts, batch_size=16)
    with torch.inference_mode():
        return text_head(torch.from_numpy(raw).to(device)).cpu().float().numpy()

# All five queries target the *water_stream* clip but in different UIQ styles.
DEMO_QUERIES = [
    ('question',   'Can you find a clip where water bubbles and splashes as it flows?'),
    ('imperative', 'Find an audio clip of loud bubbling water flowing past.'),
    ('paraphrase', 'Loud burbling and splashing as a stream flows by.'),
    ('tagging',    'water, bubbling, splashing, stream'),
    ('negative',   'Flowing water with bubbles, no human voices or machinery.'),
]

audio_emb = encode_audio(audio_paths)
query_emb = encode_text([q for _, q in DEMO_QUERIES])
audio_emb.shape, query_emb.shape

## 5. Cosine similarity & top-1 retrieval

In [ ]:
import pandas as pd

sim = query_emb @ audio_emb.T   # already L2-normalized → cosine similarity
names = [p.name for p in audio_paths]

df = pd.DataFrame(sim, index=[t for t,_ in DEMO_QUERIES], columns=names)
df.style.background_gradient(axis=1, cmap='Blues').format('{:+.3f}')

In [ ]:
for (qtype, q), row in zip(DEMO_QUERIES, sim):
    top = int(np.argmax(row))
    print(f"[{qtype:<10s}] {q}")
    print(f"  → top-1: {names[top]}  (cos={row[top]:+.3f})\n")

## 6. (Optional) Listen to the samples in the notebook

In [ ]:
from IPython.display import Audio, display
for p, cap in zip(audio_paths, captions):
    print(f'{p.name} — {cap}')
    display(Audio(str(p)))